In [1]:
# Cell 1: Install dependencies (takes ~2-3 min)
%pip install -q unsloth
%pip install -q --no-deps trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 21.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 79.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.8 MB/s eta 0:00:00
   ━━━━

In [2]:
# Cell 2: Load the base model (Qwen2.5-3B) in 4-bit
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,          # auto-detects
    load_in_4bit = True,   # fits easily on the free T4
)
print("Model loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


total weights      : 1.872 GiB
no_split classes   : ['Qwen2DecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.859 GiB requested
tied to head       : ['model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.077 GiB  free 11.875 GiB  reserve 11.859 GiB
  cuda:1  budget  13.00 GiB  weights  0.795 GiB  free 12.201 GiB  reserve 11.701 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.440 -> 12.996 GiB (memory the quantiser keeps for its own load-time buffers)
note: tied embeddings: model.embed_tokens pinned with the head so accelerate does not have to mirror the weight across devices


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully


In [3]:
# Cell 3: Load text-to-SQL dataset WITH schema included
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Total examples: {len(dataset)}")
print("\nExample:")
print("Schema:", dataset[0]["context"])
print("Question:", dataset[0]["question"])
print("SQL:", dataset[0]["answer"])

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Total examples: 78577

Example:
Schema: CREATE TABLE head (age INTEGER)
Question: How many heads of the departments are older than 56 ?
SQL: SELECT COUNT(*) FROM head WHERE age > 56


In [4]:
# Cell 4: Create train/test split
dataset = dataset.shuffle(seed=42)
test_data = dataset.select(range(100))
train_data = dataset.select(range(100, 8100))   # 8k training examples is plenty for 300 steps
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

Train: 8000, Test: 100


In [5]:
# Cell 5: Baseline - test the model BEFORE fine-tuning
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)  # switch to inference mode

def make_prompt(ex):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content":
          f"Given this database schema:\n{ex['context']}\n\n"
          f"Convert this question to a SQL query. Reply with ONLY the SQL, no explanation.\n\n"
          f"Question: {ex['question']}"}],
        tokenize=False, add_generation_prompt=True
    )

baseline_outputs = []
for i, ex in enumerate(test_data):
    inputs = tokenizer(make_prompt(ex), return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    baseline_outputs.append({"question": ex["question"], "gold": ex["answer"], "predicted": answer.strip()})
    if (i+1) % 20 == 0:
        print(f"{i+1}/100 done")

# Save results
import json
with open("baseline_results.json", "w") as f:
    json.dump(baseline_outputs, f, indent=2)
print("Baseline complete. Saved to baseline_results.json")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

20/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

40/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

60/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

80/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

100/100 done
Baseline complete. Saved to baseline_results.json


In [6]:
# Cell 6: Peek at what the untrained model produces
for r in baseline_outputs[:3]:
    print("Q:", r["question"])
    print("Gold SQL:", r["gold"])
    print("Model SQL:", r["predicted"])
    print("-" * 60)

Q: When Essendon played away; where did they play?
Gold SQL: SELECT venue FROM table_name_50 WHERE away_team = "essendon"
Model SQL: SELECT venue FROM table_name_50 WHERE away_team = 'Essendon'
------------------------------------------------------------
Q: What is the lowest numbered game against Phoenix with a record of 29-17?
Gold SQL: SELECT MIN(game) FROM table_name_61 WHERE opponent = "phoenix" AND record = "29-17"
Model SQL: SELECT MIN(game) FROM table_name_61 WHERE opponent = 'Phoenix' AND record = '29-17'
------------------------------------------------------------
Q: Who did the Texan's play on Week 4?
Gold SQL: SELECT opponent FROM table_name_37 WHERE week = "4"
Model SQL: SELECT opponent FROM table_name_37 WHERE week = '4'
------------------------------------------------------------


In [13]:
# Cell 7: Add LoRA adapters (the small trainable add-on layer)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA rank - size of the add-on layer
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA adapters attached")

Unsloth: Already have LoRA adapters! We shall skip this step.


LoRA adapters attached


In [14]:
# Cell 8: Format training data into chat conversations
def format_example(ex):
    messages = [
        {"role": "user", "content":
         f"Given this database schema:\n{ex['context']}\n\n"
         f"Convert this question to a SQL query. Reply with ONLY the SQL, no explanation.\n\n"
         f"Question: {ex['question']}"},
        {"role": "assistant", "content": ex["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_formatted = train_data.map(format_example)
print(train_formatted[0]["text"][:500])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Given this database schema:
CREATE TABLE table_name_19 (long INTEGER, gain VARCHAR, loss VARCHAR)

Convert this question to a SQL query. Reply with ONLY the SQL, no explanation.

Question: Can you tell me the lowest Long that has the Gain of 20, and the Loss smaller than 0?<|im_end|>
<|im_start|>assistant
SELECT MIN(long) FROM table_name_19 WHERE gain = 20 AND loss < 0<|im_end|>



In [15]:
# Cell 9: Train (roughly 30-60 min)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_formatted,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,          # ~2400 examples seen; enough for a strong first run
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 10,
        optim = "adamw_8bit",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()
print("Training complete!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/8000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 8,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.956547
20,0.904696
30,0.607265
40,0.591162
50,0.576186
60,0.573725
70,0.568894
80,0.576541
90,0.590596
100,0.575131


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.


Training complete!


In [17]:
# Cell 10: Test the model AFTER fine-tuning
FastLanguageModel.for_inference(model)

finetuned_outputs = []
for i, ex in enumerate(test_data):
    inputs = tokenizer(make_prompt(ex), return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    finetuned_outputs.append({"question": ex["question"], "gold": ex["answer"], "predicted": answer.strip()})
    if (i+1) % 20 == 0:
        print(f"{i+1}/100 done")

import json
with open("finetuned_results.json", "w") as f:
    json.dump(finetuned_outputs, f, indent=2)
print("After-test complete. Saved to finetuned_results.json")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

20/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

40/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

60/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

80/100 done


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

100/100 done
After-test complete. Saved to finetuned_results.json


In [19]:
# Cell 11: Score before vs after
import re, json

def normalize_sql(sql):
    sql = sql.replace("```sql", "").replace("```", "")   # strip markdown
    sql = sql.strip().rstrip(";")
    sql = re.sub(r"\s+", " ", sql)                        # collapse whitespace
    return sql.lower()

def score(results):
    exact = sum(normalize_sql(r["predicted"]) == normalize_sql(r["gold"]) for r in results)
    return exact

with open("baseline_results.json") as f:
    baseline = json.load(f)

base_score = score(baseline)
ft_score = score(finetuned_outputs)

print(f"BEFORE fine-tuning: {base_score}/100 exact matches")
print(f"AFTER  fine-tuning: {ft_score}/100 exact matches")

BEFORE fine-tuning: 3/100 exact matches
AFTER  fine-tuning: 75/100 exact matches


In [20]:
# Cell 12: Save the LoRA adapter
model.save_pretrained("qwen25-3b-text2sql-lora")
tokenizer.save_pretrained("qwen25-3b-text2sql-lora")

import shutil
shutil.make_archive("qwen25-3b-text2sql-lora", "zip", "qwen25-3b-text2sql-lora")
print("Saved and zipped")

Unsloth: Restored added_tokens_decoder metadata in qwen25-3b-text2sql-lora/tokenizer_config.json.


Saved and zipped


In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')